In [1]:
# https://light-tree.tistory.com/196
# PCGrad

**Teacher 모델**: 원본 E5
  * Output: `(B, L, D)` → token-level hidden states
  * 문장 embedding 얻으려면 token-level pooling 필요 → mask 꼭 사용

**Student 모델**: LoRA만 붙인 E5, MultiVectorHead 없음
  * Output: `(B, L, D)` → token-level hidden states
  * 문장 embedding 얻으려면 teacher처럼 **token-level pooling + mask** 사용
  * MultiVectorHead 없으면 student는 K개 vector가 아니라 단일 sentence vector

**Student 모델**: LoRA + MultiVectorHead 있는 경우
  * Output: `(B, K, D)` → 이미 pooled된 multi-vector
  * **pooling 하면 안 됨**, mask 필요 없음

| 모델                               | Output shape | Pooling 필요 | Mask 필요 |
| -------------------------------- | ------------ | ---------- | ------- |
| Teacher                          | (B, L, D)    | ✅ yes      | ✅ yes   |
| Student (LoRA only)              | (B, L, D)    | ✅ yes      | ✅ yes   |
| Student (LoRA + MultiVectorHead) | (B, K, D)    | ❌ no       | ❌ no    |

### 사전 설정

In [2]:
# Add-on Layer: 카테고리 임베딩 레이어(Category Embedding)와 이 둘을 합치는 퓨전 레이어(Linear Layer)만 새로 만듭니다.
# 이렇게 모델 구조를 뜯어고치는 건 나중에 '평점'이나 '페이지 수' 같은 숫자를 꼭 넣어야 할 때 하셔도 늦지 않습니다.
import random
import math
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from transformers import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType #, AdapterConfig

In [3]:
# book_path = './data/e5_book_meta.parquet'
book_path = './sample/book_sample.parquet'
books = pd.read_parquet(book_path)

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)            # 기본 Python random 고정
    np.random.seed(seed)         # NumPy 랜덤 고정
    torch.manual_seed(seed)      # CPU 연산 랜덤 고정
    torch.cuda.manual_seed(seed) # GPU 모든 디바이스 랜덤 고정
    torch.cuda.manual_seed_all(seed)  # 멀티 GPU일 때

    # 연산 재현성
    torch.backends.cudnn.deterministic = True  # cuDNN 연산을 determinisitc으로 강제
    torch.backends.cudnn.benchmark = False     # CUDA 성능 자동 튜닝 기능 끔 → 완전 재현 가능

set_seed(42)

In [5]:
EPOCHS = 20
WARMUP_RATIO = 0.1 # warmup = 학습 초반에 LR을 낮게 시작해서 천천히 올리는 기법 / Transformer, BERT, GPT 등에서 매우 중요

# # 일단 이건 128 배치에서 최적값인듯
LEARNING_RATE = 1e-3
BATCH_SIZE = 128

TEMPERATURE = 0.05 # 타우, 높히면 약간의 오차를 더 허용하게 됨
NEG_RATIO = 0.2 # 0.1

LAMBDA = 100.0 # 높히면 semantic, 낮추면 카테고리 분류

In [6]:
import wandb

wandb.init(
    project="retrieval-contrastive",
    name = "new_multi",
)

/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


### Transformer 내부의 특정 Layer(Wq, Wk, Wv 등)에 "LoRA 모듈"을 주입!
```
E5 Model (pretrained)
    ├── LayerNorm
    ├── 24× Transformer Layers
    │       ├── Attention (Q,K,V,O)
    │       ├── FFN Layers
    │       └── ... (원래 weight 고정)
    └── Pooler → embedding vector   # e.g., CLS embedding

LoRA Injected:
    ├── W_q = W_q + A_q B_q
    ├── W_k = W_k + A_k B_k
    ├── W_v = W_v + A_v B_v
    ├── FFN = FFN + A_ffn B_ffn
    └── (small learnable matrices only)
```

In [7]:
# import os

# print(os.listdir("."))
# if os.path.exists("intfloat"):
#     print(os.listdir("intfloat"))
# !rm -rf intfloat

In [8]:
model_name = "intfloat/e5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
teacher_model = AutoModel.from_pretrained(model_name)
teacher_model.eval()  # 평가 모드 (Dropout 등 비활성화)
for param in teacher_model.parameters():
    param.requires_grad = False # 절대 학습되지 않도록 잠금

# 선생님과 별개의 새로운 인스턴스를 만들어야 함!
student_base_model = AutoModel.from_pretrained(model_name)
lora_config = LoraConfig( # Linear(d → d) -→ Linear(d → d) + LoRA(d → d)
    task_type=TaskType.FEATURE_EXTRACTION,  # 임베딩 fine-tuning
    # LoRA가 분류기와 같은 output head에 적용되는 것이 아니라
    # 모델의 Transformer 블록(encoder)에만 적용되도록
    r=16,    # LoRA rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
base_model = get_peft_model(student_base_model, lora_config)
# 원래 weight(W)는 freeze! 훈련 과정에서 업데이트되지 않음.
# LoRA에서 추가된 A, B 행렬만 학습 학습량은 전체의 0.1%~1% 수준으로 줄어듦.
# forward 시에는 LoRA의 low-rank update가 추가됨 ex) outputs = W x + (B(Ax)) * (α / r)

Query가 Key와 얼마나 관련 있는지 계산 (similarity)
<br>=> 그 관련도(softmax)를 Value에 곱해서 Query에 맞는 정보만 뽑음
<br>=> 결과 vecter → key를 참고해서 value를 찾은 query
```
vector, _ = self.attention(
    query, → 하고픈 질문 → 학습 가능한 파라미터일수도
    key,   → 정보에 대한 설명 → Q가 무엇과 관련 있는지 계산
    value, → 찾고픈 정보 → Q·K해서 나온 값이랑 비교
)
```
의문점
<br>1. 그럼 모든 attention에서 q,k,v는 고정된 값이야? 아님 학습가능한 파라미터일수도 잇나? 아님 모델의 레이어가 바뀌면서 학습하게 되는건가
<br>2. q=k=v 일수도 있나? 어텐션의 종류에 따라 막 뭐가 같고 뭐는 다르고 할 수 있음?
<br>=> Transformer에서 q,k,v를 생성하는 방식과 머 파라미터 업데이트 방식 같은거 찾아보자

In [10]:
import torch
import torch.nn as nn

# 원래 코드처럼.....
# sequence_output = base_model(input)
# multi_vector = Attention(query_tokens, sequence_output)
# 이래 해버리면

class E5MultiVectorHead(nn.Module):
    def __init__(self, base_model, hidden_dim=384):
        super().__init__()
        self.base_model = base_model # LoRA가 적용된 E5
        # self.dropout = nn.Dropout(p=0.3)
        
        # Genre Head (Vector 0용)
        self.genre_query = nn.Parameter(  # 일단 텐서랑은 다르게 학습가능한 텐서
            torch.randn(1, 1, hidden_dim)) # 1x1xh_d 차원인데 랜덤값 넣음
        self.genre_attn = nn.MultiheadAttention(hidden_dim, 8, batch_first=True)
        
        # Content Head (Vector 1용)
        self.content_query = nn.Parameter(torch.randn(1, 1, hidden_dim))
        self.content_attn = nn.MultiheadAttention(hidden_dim, 8, batch_first=True)

    def forward(self, input_ids, attention_mask, **kwargs):
        # 1. Base Model (E5) 통과 -> 모든 토큰의 정보 획득
        # E5 같은 인코더는 토큰 단위로 문장을 표현하고, d 차원의 벡터로 변환함
        outputs = self.base_model( # output은 모델이 출력한 객체임! 모델쓰는거마다 다름
            input_ids=input_ids, # (B, L) 한번에 B개의 문장, 각 문장은 L의 토큰 길이
            attention_mask=attention_mask
        )
        sequence_output = outputs.last_hidden_state # (B, L, D) D는 각 토큰의 벡터 차원임
        batch_size = input_ids.shape[0]
        key_padding_mask = ~attention_mask.bool()

        # 2. Attention Pooling
        # 2.1. Genre Path: Detach 적용! => Encoder는 장르 로스의 영향을 안 받음
        seq_detached = sequence_output.detach() # gradient 끊기
        # seq_detached = self.dropout(seq_detached)
        genre_q = (
            self.genre_query # 아까 랜덤으로 만든 쿼리 토큰을
            .repeat(batch_size, 1, 1) # repeat 만큼 확장함
        ) # => (B, 1, D)
        v0, _ = self.genre_attn( # (B, 1, D)
            genre_q,      # 우리가 궁금한 주제
            seq_detached, # 책의 전체 내용
            seq_detached, # 책의 전체 내용
            key_padding_mask=key_padding_mask
        )
        v0 = F.normalize(v0, p=2, dim=2) # Contrastive Learning을 위해 필수
        
        # 2.2. Content Path: Encoder 학습 가능
        seq_content = sequence_output
        content_q = self.content_query.repeat(batch_size, 1, 1) # (B, 1, D)
        v1, _ = self.content_attn( # (B, 1, D)
            content_q,
            seq_content,
            seq_content,
            key_padding_mask=key_padding_mask
        )
        v1 = F.normalize(v1, p=2, dim=2) # Contrastive Learning을 위해 필수

        return v0.squeeze(1), v1.squeeze(1)

### 데이터 불러오기

In [11]:
def build_text(row): # 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
    parts = [
        f"Title: {row['title']} |",
        # f"Category: {row['category']} |", # oracle
        f"Description: {row['description']}"
    ]
    return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
        [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
    ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Category: ... Description: ..."

books["text"] = books.apply(build_text, axis=1) # 새 컬럼 text에 대해서.... 문장 만듦

# 100개 미만인 카테고리는 노이즈로 간주하고 삭제
counts = books['category'].value_counts()
valid_categories = counts[counts > 100].index
books = books[books['category'].isin(valid_categories)]

In [12]:
dataset = Dataset.from_pandas(books)

le = LabelEncoder()
le.fit(dataset['category'])   # 전체 데이터로 학습

def encode_label(x):
    return {"label": le.transform([x["category"]])[0]}

dataset = dataset.map(encode_label)

num_classes = len(le.classes_)

Map:   0%|          | 0/81845 [00:00<?, ? examples/s]

collate_fn은 raw text와 label을 텐서로 묶어 모델이 학습할 수 있는 형태로 만들어줌
DataLoader는 이 함수로 미리 전처리한 batch를 모델에 공급하는 역할을 함
```
Dataset row(dict)
     ↓ (DataLoader)
batch = [row1, row2, ...] (list)
     ↓ (collate_fn)
텍스트 리스트 + 라벨 리스트
     ↓ (tokenizer)
input_ids, attention_mask (tensor)
     ↓
(inputs, labels)
     ↓
model(**inputs)
```

In [13]:
# Transformer 모델은 이런 raw 텍스트를 바로 처리 못 하고
# 토크나이저를 거쳐 tensor(batch_input_ids, batch_attention_mask) 형태가 필요함.
def collate_fn(batch): # DataLoader가 batch마다 호출
    # texts = [f"passage: {x['text']}" for x in batch]
    texts = [f"query: {x['text']}" for x in batch]
    labels = torch.tensor([x['label'] for x in batch])  # 라벨을 int 리스트 → torch.tensor 로 변환

    """
    토크나이저:
    텍스트를 token id로 변환 (input_ids), attention_mask 생성,
    batch의 최대 length에 맞춰 패딩, 출력 타입은 PyTorch tensor

    { 'input_ids': tensor([[101,  ... , 102], ...]),
      'attention_mask': tensor([[1,1,1,0,0...], ...) }
    """
    inputs = tokenizer(
      texts, padding=True, truncation=True, max_length=256, return_tensors="pt")

    return inputs, labels

In [14]:
total_len = len(dataset)
train_len = int(total_len * 0.8)
valid_len = total_len - train_len

train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

지금의 구조는 contrastive loss, genre/content KD, dynamic α, GradNorm, multi-vector head 등이 서로 얽혀 있어 LR 변화에 매우 민감함

- contrastive(genre) 비중이 큰 α=0.8 상태
- genre/content 양방향 KD 압력
- multi-vector head 정렬 전
- GradNorm 비율이 아직 수렴 안 됨
- α decay가 아직 적용 전(장르 비중 과다)

warmup을 너무 짧게(0\~2%) 주면 learning rate가 지나치게 빨리 상승해 초기 단계에서 gradient explosion이나 embedding 붕괴가 일어나고, 반대로 너무 길게(10\~20%) 주면 LR이 천천히 올라가는 동안 contrastive 쪽 표현 학습이 지연되고 embedding collapse 위험이 커져 최종 성능(MRR, top-1)까지 떨어짐

스케쥴러 자체도 바꿨는데, Linear decay는 warmup 이후에 LR이 직선으로 급격히 떨어짐. 이는 학습 후반부 KD alignment와 contrastive alignment의 미세 조정을 막아 학습이 사실상 멈추는 문제가 있음. 반면 Cosine 스케줄러는 warmup 이후 LR을 완만한 곡선 형태로 감소시키기 때문에, 초반에는 안정성을 제공하고, 중반에는 충분한 LR을 유지하며, 후반에도 작은 폭이지만 의미 있는 업데이트를 이어갈 수 있을 것임

### utiliy

In [15]:
def compute_retrieval_accuracy(model, dataloader, device, k=10, alpha=0.4):
    """
    [모델 A] 검증 함수
    - v0 (Genre), v1 (Content), Hybrid (Weighted Sum) 세 가지 성능을 모두 측정
    - alpha: Hybrid 점수 계산 시 v0의 가중치 (기본 0.4)
    """
    v0_list = []
    v1_list = []
    labels_list = []

    model.eval()
    
    # 1. 데이터 수집 (Inference)
    with torch.no_grad():
        for batch_inputs, labels in dataloader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            labels = labels.to(device)

            # [변경점] 모델이 이제 v0, v1 두 개를 뱉습니다!
            v0, v1 = model(**batch_inputs)

            # CPU로 옮겨서 리스트에 저장 (GPU 메모리 아끼기)
            v0_list.append(v0.cpu())
            v1_list.append(v1.cpu())
            labels_list.append(labels.cpu())

    # 2. 전체 데이터 합치기
    all_v0 = torch.cat(v0_list, dim=0)   # (N, Dim)
    all_v1 = torch.cat(v1_list, dim=0)   # (N, Dim)
    all_labels = torch.cat(labels_list, dim=0) # (N)

    # ---------------------------------------------------------
    # 3. 내부 함수: 임베딩과 라벨을 주면 성능(MRR 등)을 계산해주는 함수
    # ---------------------------------------------------------
    def calc_metrics(embeddings, labels, desc=""):
        # 유사도 행렬 (N, N)
        similarity = torch.matmul(embeddings, embeddings.T)
        # 자기 자신 제외 (-무한대)
        similarity.fill_diagonal_(-1e9)

        # Top-K 추출
        _, topk_idx = similarity.topk(k, dim=1)
        nn_labels = labels[topk_idx] # (N, k)

        # 정답 여부 (Broadcasting)
        # 내 라벨과 이웃 라벨이 같으면 정답(True)
        hits = (nn_labels == labels.unsqueeze(1))

        # MRR 계산
        ranks = hits.float()
        reciprocal_rank = []
        for i in range(ranks.size(0)):
            # 정답인 위치(index) 찾기
            pos_positions = torch.nonzero(ranks[i]).flatten()
            if len(pos_positions) == 0:
                reciprocal_rank.append(0.0)
            else:
                # 가장 가까운 정답의 순위 (0-index니까 +1)
                reciprocal_rank.append(1.0 / (pos_positions[0].item() + 1))
        
        mrr = sum(reciprocal_rank) / len(reciprocal_rank)
        
        # Precision@K
        precision = hits.float().mean().item()
        
        return {"mrr": mrr, "precision@k": precision}

    # ---------------------------------------------------------
    # 4. 각 벡터별 성능 측정
    # ---------------------------------------------------------
    
    # (A) Vector 0 성능 (장르 군집화 능력) -> MRR이 매우 높아야 정상
    metrics_v0 = calc_metrics(all_v0, all_labels, desc="Genre(v0)")

    # (B) Vector 1 성능 (내용 기반 장르 일치도) -> v0보단 낮지만 높아야 함
    metrics_v1 = calc_metrics(all_v1, all_labels, desc="Content(v1)")

    # (C) ★ Hybrid 성능 (실제 서비스 랭킹) ★
    # 가중 합: Score = alpha * sim(v0) + (1-alpha) * sim(v1)
    # 내적으로 전개하면: alpha * (v0 @ v0.T) + (1-alpha) * (v1 @ v1.T)
    # 벡터 자체를 섞는 게 아니라, 유사도 점수를 섞는 방식이 더 정확함
    sim_v0 = torch.matmul(all_v0, all_v0.T)
    sim_v1 = torch.matmul(all_v1, all_v1.T)
    
    final_sim_score = (alpha * sim_v0) + ((1 - alpha) * sim_v1)
    
    # Hybrid Metric 계산 로직 (calc_metrics 함수 재사용 불가 - sim matrix가 이미 있어서)
    final_sim_score.fill_diagonal_(-1e9)
    _, topk_idx_h = final_sim_score.topk(k, dim=1)
    nn_labels_h = all_labels[topk_idx_h]
    hits_h = (nn_labels_h == all_labels.unsqueeze(1))
    
    # Hybrid MRR (복붙)
    ranks_h = hits_h.float()
    reciprocal_rank_h = []
    for i in range(ranks_h.size(0)):
        pos_positions = torch.nonzero(ranks_h[i]).flatten()
        if len(pos_positions) == 0: reciprocal_rank_h.append(0.0)
        else: reciprocal_rank_h.append(1.0 / (pos_positions[0].item() + 1))
    mrr_hybrid = sum(reciprocal_rank_h) / len(reciprocal_rank_h)

    # 5. 결과 리턴
    return {
        "v0_mrr": metrics_v0["mrr"],          # 장르 필터링 능력
        "v1_mrr": metrics_v1["mrr"],          # 내용 벡터의 품질
        "hybrid_mrr": mrr_hybrid,             # ★ 최종 성능 (가장 중요) ★
        "v0_prec": metrics_v0["precision@k"],
        "hybrid_prec": hits_h.float().mean().item()
    }

In [16]:
# 일단 한번 해보고 개선되면 memory bank 식으로 바꿔보자
# 근데 배치가 크면 굳이 memory bank로 안바꿔도 된다는디?
def v0_negative(embeddings, labels, neg_ratio=0.1): # k=3):
    batch_size = embeddings.size(0)
    k = max(3, int(batch_size * neg_ratio)) # 배치 사이즈에 따라 K값 결정

    # 1. 유사도 행렬 계산
    similarity = torch.matmul(embeddings, embeddings.T) # (B, B)

    # hard_neg_sims = []
    # for i in range(batch_size): # for문 대신 행렬 연산(Vectorization)으로 바꾸면 빨라진대
    #     mask = labels != labels[i]           # 다른 라벨만
    #     sim_row = similarity[i].clone()
    #     sim_row[~mask] = -1e9                # 같은 라벨 제외
    #     topk_sim, _ = sim_row.topk(k)        # 가장 가까운 k개
    #     hard_neg_sims.append(topk_sim)
    # hard_neg_sims = torch.stack(hard_neg_sims, dim=0)  # (batch_size, k)

    # 2. Positive Mask 생성 (같은 라벨인 곳 True)
    # labels: (N) -> (N, 1) == (1, N) 브로드캐스팅
    pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)

    # 3. 같은 라벨(Positive + 자기자신)은 유사도를 -무한대로 보내서 topk에서 제외
    # clone()을 안 쓰면 원본 similarity가 망가지므로 주의 (필요시)
    sim_for_neg = similarity.clone()
    sim_for_neg.masked_fill_(pos_mask, -1e9)  # 같은 라벨 제외

    # 4. 각 행(Query)마다 가장 높은 유사도를 가진 k개(Hard Negatives) 추출
    hard_neg_sims, _ = sim_for_neg.topk(k, dim=1) # (N, k)

    return hard_neg_sims

def supcon_loss(embeddings, labels):
    # https://www.blossominkyung.com/deeplearning/contrastive-learning
    # 같은 카테고리는 가까이, 다른 카테고리는 멀리
    similarity = torch.matmul(embeddings, embeddings.T)

    # Mask 생성
    labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
    identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
    labels_eq = labels_eq & (~identity_mask)

    pos_mask = labels_eq.float()   # (N, N)

    """
    anchor: 닻(기준점)
    pos_sim: anchor와 pos 샘플들의 유사도 벡터
    neg_sim: anchor와 neg 샘플들의 유사도 벡터
    """
    pos_sim = similarity * pos_mask
    neg_sim = v0_negative(embeddings, labels, neg_ratio=NEG_RATIO) # (N, k) → anchor i의 negative k개

    # loss 확대: 정답(0.8/0.05=16), 오답(0.7/0.05=14) => exp(16) ≈ 8,886,110 vs exp(14) ≈ 1,202,604 7배 이상 차이남
    # => 0.1 차이도 크게 만들어 모델이 pos를 더더더 1에 가깝도록 맞춤
    pos_sim = pos_sim / TEMPERATURE # 아니 근데 생각해보니까 noise는 /temp 전에 들어가잖아?
    neg_sim = neg_sim / TEMPERATURE

    # 모든 positive를 합산해서 ratio 계산 → 카테고리별 embedding을 한 점에 모으는 경향
    loss = -torch.log(
        torch.exp(pos_sim).sum(dim=1) /
        (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
    ).mean()
    # sum해주는 이유는.. 한 anchor(기준 샘플)당 여러 개의 pos/neg 쌍이 존재할 수 있기 때문
    # 그림 mean은 왜 하는거지... 배치내 모든 loss를 평균 내는것임! -> 이번 배치의 loss는 0.36다~

    return loss

In [17]:
def v1_negative(embeddings, labels, neg_ratio=0.1): # k=3):
    batch_size = embeddings.size(0)
    k = max(3, int(batch_size * neg_ratio)) # 배치 사이즈에 따라 K값 결정

    # 1. 유사도 행렬 계산
    similarity = torch.matmul(embeddings, embeddings.T) # (B, B)

    # 2. Self Masking 나 자신(대각전)만 제외하려고 마스킹
    mask_self = torch.eye(batch_size, device=embeddings.device).bool()
    sim_for_neg = similarity.clone() # clone()안하면 원본 similarity가 망가짐
    sim_for_neg.masked_fill_(mask_self, -1e9) # 마스크 보고 -무한대 설정, topk에서 제외됨
    
    # 3. 비슷한 거 Top-K 추출 = hard_neg
    # 반환값: 점수(values)가 아니라 '인덱스(indices)'를 뽑아야 임베딩을 가져옴
    _, topk_indices = similarity.topk(k, dim=1) # (B, K)
    
    # 4. 임베딩 가져오기 (Gather)
    # (B, K) 인덱스를 이용해 실제 (B, K, Dim) 벡터를 만듦
    # expand를 위해 차원 맞추기
    # embeddings: (B, Dim) -> (B, K, Dim)으로 모으기 위한 과정
    neg_embeddings = embeddings[topk_indices] # (B, K, Dim) -> PyTorch의 고급 인덱싱

    return neg_embeddings
    
def infonce_loss(query, positive_key, negatives, temperature=0.05):
    """
    query: (B, Dim) - v1_content
    positive_key: (B, Dim) - v1_content (Self) or Pair
    negatives: (B, K, Dim) - 위에서 뽑은 Hard Negatives
    """
    
    # 1. Positive Similarity(점수): 자기 자신과의 내적
    pos_sim = torch.sum(query * positive_key, dim=1, keepdim=True) # (B, 1) 
    
    # 2. Negative Similarity (Hard Negatives)
    neg_sim = torch.matmul(
        query.unsqueeze(1), # (B, D) -> (B, 1, D)
        negatives.transpose(1, 2) # (B, K, D) -> (B, D, K)
    ).squeeze(1) # (B, 1, D)*(B, D, K) = (B, (1,D)*(D,K)) = (B, 1, K) -> (B, K)

    # 3. In-batch Negatives (Optional, but recommended)
    # 배치 내 다른 샘플들도 Negative로 활용
    batch_neg_sim = torch.matmul(query, positive_key.T) # (B, B)
    mask = torch.eye(query.size(0), device=query.device).bool()
    batch_neg_sim = batch_neg_sim.masked_select(~mask).view(query.size(0), -1) # (B, B-1)
    
    # 4. 로짓 합치기: [Positive | Negatives] -> (B, 1+K)
    logits = torch.cat([pos_sim, neg_sim], dim=1)
    logits = logits / temperature
    
    # 5. Cross Entropy
    # 정답은 항상 0번째 인덱스 (pos_sim이 맨 앞에 있으니까)
    labels = torch.zeros(logits.size(0), dtype=torch.long, device=query.device)
    
    loss = F.cross_entropy(logits, labels)
    
    return loss

In [18]:
# --------------------------------------------------------------------------
# 1. GradNorm 계산 함수 정의
# --------------------------------------------------------------------------
def calc_grad_norm(loss, model_layer):
    """
    특정 Loss가 특정 레이어(model_layer)의 파라미터에 가하는
    Gradient의 총량(Norm)을 계산합니다.
    """
    # 1. 해당 레이어의 파라미터만 가져옴 (requires_grad=True인 것만)
    params = [p for p in model_layer.parameters() if p.requires_grad]

    if not params:
        return 0.0

    # 2. Gradient 계산 (create_graph=False, retain_graph=True 필수!)
    # retain_graph=True: 뒤에 진짜 backward()를 또 해야 하므로 그래프를 날리면 안 됨
    grads = torch.autograd.grad(
        loss,
        params,
        retain_graph=True,
        allow_unused=True
    )

    # 3. Norm(크기) 합산 (L2 Norm)
    total_norm = 0.0
    for g in grads:
        if g is not None:
            total_norm += g.pow(2).sum().item()

    return total_norm ** 0.5

### 학습루프

In [19]:
total_steps = len(train_loader) * EPOCHS

model = E5MultiVectorHead(base_model, 384)

device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_model.to(device)
model.to(device)



# 1. 파라미터를 분류해라
# - base_params: 이미 똑똑한 E5 (LR 낮게)
# - head_params: 멍청한 Query랑 Head (LR 높게)
base_params = [p for n, p in model.named_parameters() if "base_model" in n]
head_params = [p for n, p in model.named_parameters() if "base_model" not in n]

# 2. 옵티마이저에 따로따로 먹여라
optimizer = optim.AdamW([
    {'params': base_params, 'lr': 1e-4},  # 조심조심
    {'params': head_params, 'lr': 1e-3}   # 팍팍! (보통 10배 더 줌)
])


# optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# AdamW 옵티마이저로 LoRA 파라미터만 학습
# LoRA 덕분에 실제 업데이트되는 파라미터는 전체의 1% 정도

# scheduler = get_linear_schedule_with_warmup(
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.05), # WARMUP_RATIO), 원래 linear 일때 썻던 값은데 0.1이었거든? 0.05로 줄이래
    num_training_steps=total_steps,
)

In [20]:
print("Before:", model.genre_query[0, 0, :5])

Before: tensor([-1.0939,  0.7513, -0.4568,  0.2858, -1.1972], device='cuda:0',
       grad_fn=<SliceBackward0>)


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    genre_supcon = 0         # 디버깅
    genre_kd = 0         # 디버깅
    content_kd = 0         # 디버깅
    content_infonce = 0         # 디버깅
    genre_loss = 0           # 디버깅
    content_loss = 0         # 디버깅
    
    for step, (batch_inputs, labels) in enumerate(tqdm(train_loader, desc = f"Epoch: {epoch+1}")):
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        # batch_inputs = {"input_ids": tensor_of_input_ids,
        #                 "attention_mask": tensor_of_attention_mask}
        labels = labels.to(device)

        # Teacher (Original E5) Forward
        with torch.no_grad(): # 원본 모델에게 물어봅니다. "너라면 이거 어디에 배치할래?"
            teacher_outputs = teacher_model(**batch_inputs)

            # teacher_embeddings = teacher_outputs.last_hidden_state.mean(dim=1)
            hidden = teacher_outputs.last_hidden_state       # (B, L, D)
            mask = batch_inputs['attention_mask'].unsqueeze(-1)  # (B, L, 1)
            teacher_embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

            # Teacher: 정규화 필수 (보통 Mean Pooling 직후엔 길이가 1이 아님)
            teacher_norm = F.normalize(teacher_embeddings, p=2, dim=1)

        # Student LoRA Forward
        v0_genre, v1_content = model(**batch_inputs) # 딕셔너리 언패킹(**)

        # Loss 1: Genre
        loss_v0_supcon = supcon_loss(v0_genre, labels)
        loss_v0_kd = F.mse_loss(v0_genre, teacher_norm) # 테스트!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        # Loss 2: Content
        loss_v1_kd = F.mse_loss(v1_content, teacher_norm)
        
        hard_neg_embs = v1_negative(v1_content, labels)
        loss_v1_infonce = infonce_loss(v1_content, v1_content, hard_neg_embs)

        # if epoch < 1: lambda_kd = 100.0 
        # elif epoch < 3: lambda_kd = 10.0
        # else: lambda_kd = 1.0 
        # loss_content = loss_v1_infonce + (lambda_kd * loss_v1_kd)
        
        # 3. 최종 합산
        # overall_loss = loss_genre + loss_content
        
        # 테스트!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # V0 KD 비율은 1.0~10.0 정도면 충분 (나침반 역할만 하면 됨)
        # V1 KD 비율은 Strong하게 (100.0) 유지 (Encoder 지진 방지)
        loss_v0_total = loss_v0_supcon + (LAMBDA * loss_v0_kd)
        loss_v1_total = loss_v1_infonce + (0.1*LAMBDA * loss_v1_kd)
        
        overall_loss = loss_v0_total + loss_v1_total
        # 테스트!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


        # 역전파 과정
        # total_loss.backward()    # 기울기(Gradient) 계산 완료
        # optimizer.zero_grad()    # 기울기를 0으로 초기화 (삭제) 😱
        # optimizer.step()         # 0이 된 기울기로 가중치 업데이트 (변화 없음)
        optimizer.zero_grad()   # 1. 이전 기울기 청소 이거 맨 위로 옮김
        overall_loss.backward() # 2. 현재 기울기 계산
        optimizer.step()        # 3. 업데이트

        scheduler.step() # 스케줄러가 step 단위라면 여기, epoch 단위면 loop 밖으로

        total_loss += overall_loss.item()
        genre_supcon += loss_v0_supcon.item()        # 디버깅
        genre_kd += loss_v0_kd.item()                # 디버깅
        content_kd += loss_v1_kd.item()              # 디버깅
        content_infonce += loss_v1_infonce.item()    # 디버깅
        genre_loss += loss_v0_total.item()           # 디버깅
        content_loss += loss_v1_total.item()         # 디버깅

    train_loss = total_loss / len(train_loader)
    total_genre_supcon = genre_supcon / len(train_loader)            # 디버깅
    total_genre_kd = genre_kd / len(train_loader)            # 디버깅
    total_content_kd = content_kd / len(train_loader)            # 디버깅
    total_content_infonce = content_infonce / len(train_loader)  # 디버깅
    total_genre_loss = genre_loss / len(train_loader)  # 디버깅
    total_content_loss = content_loss / len(train_loader)  # 디버깅

    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train/genre_supcon": total_genre_supcon,
        "train/genre_kd": total_genre_kd,
        "train/content_kd": total_content_kd,
        "train/content_infonce": total_content_infonce,
        "train/genre_loss": total_genre_loss,
        "train/content_loss": total_content_loss,
        "valid/v0_mrr": metrics["v0_mrr"],            # 장르 필터링 능력
        "valid/v1_mrr": metrics["v1_mrr"],            # 내용 벡터의 품질
        "valid/hybrid_mrr": metrics["hybrid_mrr"],    # ★ 최종 성능 (가장 중요) ★
        "valid/v0_prec": metrics["v0_prec"],
        "valid/hybrid_prec": metrics["hybrid_prec"],
        "lr": scheduler.get_last_lr()[0],
    })
    alpha = 0.4
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"📊 [Evaluation Result] Alpha={alpha} (Genre Weight)")
    print("-" * 60)
    
    # 1. Vector 0 (Genre): 장르 필터링 담당 (점수가 아주 높아야 정상, 0.95+)
    print(f"🛡️  [Genre Filter (V0)] MRR: {metrics['v0_mrr']:.4f} | Prec@K: {metrics['v0_prec']:.4f}")
    
    # 2. Vector 1 (Content): 순수 내용 검색 담당 (V0보단 낮지만 준수해야 함)
    print(f"🎯  [Content Only(V1)] MRR: {metrics['v1_mrr']:.4f} (Reference Only)")
    
    print("-" * 60)
    
    # 3. Hybrid: 실제 서비스 성능 (이게 0.75 넘으면 SOTA급!)
    print(f"🚀  [Hybrid (Final)]   MRR: {metrics['hybrid_mrr']:.4f} | Prec@K: {metrics['hybrid_prec']:.4f}")
    print("=" * 60)

In [ ]:
print("After: ", model.genre_query[0, 0, :5])

v0_mrr (장르 점수):

목표치: 0.95 ~ 0.99 (거의 1.0에 가까워야 함)

의미: "판타지/역사 구분은 내가 신이다."

만약 이게 낮다면? → loss_genre (SupCon) 학습이 덜 된 것.

v1_mrr (내용 점수):

목표치: 0.70 ~ 0.80 (v0보단 낮음)

의미: "내용만 봐도 얼추 장르는 맞추네?"

이게 너무 낮으면(0.5 이하)? → Encoder가 장르 정보를 너무 잃어버림 (Teacher KD 비중을 살짝 높여야 함).

hybrid_mrr (최종 점수):

목표치: 0.75 + (SOTA 도전)

의미: "장르 필터링(v0) + 정밀 검색(v1)의 조화"

이 점수가 님이 최종적으로 논문에 쓰거나 자랑할 점수입니다.

Tip: alpha 값을 0.3, 0.5, 0.7 바꿔가면서 언제 점수가 제일 높은지 찾아보세요!

In [ ]:
import os
save_path = f"./new_multi_vec_{LAMBDA}_32ep.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)

### 디버깅

In [ ]:
import torch
import torch.nn.functional as F

def compute_multivector_diagnostics(model, l_cont, l_kd_genre, l_kd_content, lambda_genre=50.0, lambda_content=500.0):
    """
    Multi-Vector 학습의 '힘의 균형'을 분석하는 도구
    """
    # 1. 학습 대상 파라미터만 추출 (LoRA + Head)
    # requires_grad=True인 것만 가져와야 None 에러가 안 남
    params = [p for p in model.parameters() if p.requires_grad]

    # 2. 각각의 Loss에 대한 Gradient 추출 (Retain Graph 필수!)
    # 메모리를 좀 먹으니, 진단은 100 step마다 한 번씩만 하길 권장
    grads_cont = torch.autograd.grad(l_cont, params, retain_graph=True, allow_unused=True)
    grads_kd_g = torch.autograd.grad(l_kd_genre, params, retain_graph=True, allow_unused=True)
    grads_kd_c = torch.autograd.grad(l_kd_content, params, retain_graph=True, allow_unused=True)

    # 3. Flatten Helper
    def get_flat_grad(grads):
        vec = []
        for g in grads:
            if g is not None:
                vec.append(g.view(-1))
        return torch.cat(vec) if vec else None

    v_cont = get_flat_grad(grads_cont)
    v_kd_g = get_flat_grad(grads_kd_g)
    v_kd_c = get_flat_grad(grads_kd_c)

    if v_cont is None or v_kd_g is None or v_kd_c is None:
        return {}

    # -------------------------------------------------------
    # 4. 분석: 힘의 크기 (Magnitude) 비교
    # -------------------------------------------------------

    # A. 장르 벡터의 추진력 vs 저항력
    # Norm 비교: ||G_cont|| vs ||50 * G_kd_genre||
    norm_cont = v_cont.norm().item()
    norm_drag = v_kd_g.norm().item() * lambda_genre

    # B. 내용 벡터의 보존력
    # Norm 비교: ||500 * G_kd_content||
    norm_anchor = v_kd_c.norm().item() * lambda_content

    # -------------------------------------------------------
    # 5. 분석: 힘의 방향 (Direction) 비교
    # -------------------------------------------------------

    # A. 장르 내부 갈등 (Classification vs Regularization)
    # 음수가 나오면 서로 반대 방향 (정상)
    cos_friction = F.cosine_similarity(v_cont.unsqueeze(0), v_kd_g.unsqueeze(0)).item()

    # B. 메인 충돌 (Classification vs Content Preservation)
    # 이게 가장 중요함. 서로 반대 방향일 때 힘의 크기가 비슷해야 붕괴를 막음.
    cos_conflict = F.cosine_similarity(v_cont.unsqueeze(0), v_kd_c.unsqueeze(0)).item()

    return {
        "Force(Cluster)": norm_cont,       # 클러스터링 하려는 힘
        "Force(Drag)": norm_drag,          # 장르 이탈 막는 약한 힘 (50)
        "Force(Anchor)": norm_anchor,      # 내용 지키는 강한 힘 (500)
        "Cos(Friction)": cos_friction,     # 장르 내부 각도
        "Cos(Conflict)": cos_conflict      # 장르 vs 내용 각도
    }

grad_logs = {
    "force_cluster": [],
    "force_drag": [],
    "force_anchor": [],
    "cos_friction": [],
    "cos_conflict": [],
    "step": []
}
lora_params = [p for p in model.parameters() if p.requires_grad]

In [ ]:
import matplotlib.pyplot as plt

"""
grad_logs 딕셔너리에 저장된 Multi-Vector 학습 진단 데이터를 시각화합니다.
Keys expected: 'step', 'cos_friction', 'cos_conflict',
               'force_cluster', 'force_drag', 'force_anchor'
"""
steps = grad_logs["step"]

# 데이터를 CPU로 옮기는 헬퍼 함수
def to_cpu(data_list):
    return [x.item() if torch.is_tensor(x) else x for x in data_list]

# 그래프 스타일 설정 (가독성 향상)
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12), sharex=True)

# ---------------------------------------------------------
# 1. Cosine Similarity (방향성) 그래프
# ---------------------------------------------------------
# Cos(Friction): 장르 벡터 내부의 방향 (Cont vs KD_Genre)
ax1.plot(steps, grad_logs["cos_friction"],
         label="Cos(Friction): Genre Internal", color='green', alpha=0.7, linestyle='--')

# Cos(Conflict): 메인 충돌 (Genre vs Content) -> 이게 가장 중요!
ax1.plot(steps, grad_logs["cos_conflict"],
         label="Cos(Conflict): Genre vs Content", color='red', linewidth=2)

# 기준선 (0.0)
ax1.axhline(0, color='black', linewidth=1, linestyle='-')

ax1.set_title("1. Gradient Direction Analysis (Cosine Similarity)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Cosine Similarity (-1 ~ 1)", fontsize=12)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.6)

# ---------------------------------------------------------
# 2. Gradient Norms (힘의 크기) 그래프
# ---------------------------------------------------------
# Force(Cluster): 장르를 나누려는 힘 (Cont)
ax2.plot(steps, grad_logs["force_cluster"],
         label="Force(Cluster): ||G_cont||", color='blue', linewidth=2)
# 2. Anchor (Orange) - 내용을 지키려는 힘 (500 * KD_Content) -> 빨간색이 파란색을 감싸야 함
ax2.plot(steps, to_cpu(grad_logs["force_anchor"]),
         label="Force(Anchor): ||λ*α*G_kd_c||", color='orange', linewidth=2, linestyle='-')

# 3. Drag (Green) 장르 이탈을 막는 약한 힘 (50 * KD_Genre) -> 바닥에 깔려야 함
ax2.plot(steps, to_cpu(grad_logs["force_drag"]),
         label="Force(Drag): ||λ*(1-α)*G_kd_g||", color='green', alpha=0.6, linestyle=':')

ax2.set_title("2. Gradient Magnitude Analysis (Forces)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Training Step", fontsize=12)
ax2.set_ylabel("Gradient Norm", fontsize=12)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()